# 예제 10. 금융업무 통합 자동화 에이전트
### 여러 금융업무를 하나의 에이전트로 연결해 일괄 처리 · 2일 차 · 40분

> **실행 방법**: 위에서부터 순서대로 실행하세요. 셀을 클릭하고 `Shift + Enter`. AI 답변이 나오는 셀은 몇 초~수십 초 걸립니다. 셀 왼쪽이 `[*]`이면 아직 실행 중입니다.

| 항목 | 내용 |
|---|---|
| 데이터 | `data/bank_customers.csv` (+ 직접 만든 고객 일부 파일) |
| 오늘의 결과물 | 한 번 실행으로 Excel + 차트 3종 + Word 보고서 일괄 생성 |

**오늘 해결할 질문**: "파일과 분석 목적만 넣으면, 보고서 묶음이 한 번에 나올까?"

## 0. 실습 준비

FinanceAgent를 불러오고, 이 PC의 로컬 AI(Ollama)가 켜져 있는지 확인합니다.
`AI 연결 확인: qwen3:4b-instruct 준비 완료`가 나오면 성공입니다.

In [ ]:
import json, urllib.request
from pathlib import Path
from finance_agent import FinanceAgent
import llm_config

DATA = "data/bank_customers.csv"
models = [m["name"] for m in json.load(urllib.request.urlopen("http://127.0.0.1:11434/api/tags"))["models"]]
print("데이터 파일:", Path(DATA).is_file())
print("AI 연결 확인:", llm_config.OLLAMA_MODEL, "준비 완료" if llm_config.OLLAMA_MODEL in models else "→ 모델이 없습니다. 강사에게 알려 주세요")
agent = FinanceAgent(provider="ollama")

## 10-1. 통합 흐름 한눈에

```
파일 + 분석 목적
   ↓
[① 입력 검증] → [② 고객군 5개 기준 가입률 계산] → [③ AI: 분석 요약 · 특징 · 마케팅 전략 초안] → [④ 결과보기]
                                                                                              ├─ Excel (5개 시트)
                                                                                              ├─ 차트 3종 (PNG · HTML)
                                                                                              └─ Word 보고서
```

예제 07 · 08 · 09에서 만든 기능을 그대로 이어 붙였습니다. 교육생이 바꾸는 것은 **파일과 분석 목적** 두 가지뿐입니다.

## 10-2. 한 번에 실행하기 ⭐

In [ ]:
결과 = agent.run_workflow(
    file=DATA,
    task="고객군별 정기예금 가입률 분석 및 보고서 작성",
)

> 💬 AI 답변의 문장은 PC와 모델에 따라 조금씩 다릅니다. 표·숫자·파일은 Python이 계산하므로 모두 같게 나옵니다.

## 10-3. 만들어진 결과물 한눈에

In [ ]:
from IPython.display import Image, display

for f in 결과["artifacts"]:
    print(f"{Path(f).stat().st_size / 1024:8.0f} KB  {f}")
for f in 결과["artifacts"]:
    if f.endswith(".png"):
        display(Image(filename=f, width=480))

## 10-4. 🔧 다른 업무 시나리오로 실행 (실습)

**분석 목적**만 바꿔 다시 실행합니다. 계산과 파일 형식은 같고, AI가 쓰는 요약 · 전략이 목적에 맞게 바뀝니다.

In [ ]:
결과 = agent.run_workflow(
    file=DATA,
    task="60대 이상 · 은퇴 고객 대상 정기예금 안내 전략 검토",     # ← 바꿔 보세요
)

## 10-5. [선택] 🔧 다른 고객 파일로 실행

주택담보대출이 있는 고객만 뽑아 같은 통합 실행을 합니다. 결과 파일 이름 끝에 파일 이름이 붙어 따로 저장됩니다.

In [ ]:
import pandas as pd

원본 = pd.read_csv(DATA)
원본[원본["housing"] == "yes"].to_csv("data/bank_customers_housing.csv", index=False)

결과 = agent.run_workflow(file="data/bank_customers_housing.csv", task="주택담보대출 보유 고객의 정기예금 가입률 분석 및 보고서 작성")

## 10-6. [선택] 에이전트 속을 들여다보기

FinanceAgent 안에서 돌아가는 LangGraph 워크플로의 구조를 글자로 출력합니다. 네 단계가 한 줄로 이어진 것을 확인하세요.

In [ ]:
from agent_core.graph import build_agent

print(build_agent("ex10").get_graph().draw_mermaid())

## 10-7. 우리 회사에서 쓰려면 ⚠️

| 확인할 것 | 내용 |
|---|---|
| 승인된 데이터 | 고객정보는 내부 승인 범위 안에서만, 가명·집계 데이터 우선 |
| 승인된 AI 환경 | 로컬 AI 또는 회사가 승인한 AI만 사용 (외부 AI에 내부자료 입력 금지) |
| 숫자 검증 | 계산은 Python, AI 문장 속 숫자는 자동 검증 + 사람 대조 |
| 최종 책임 | AI 결과물은 초안. 보고 · 의사결정 전 담당자가 검토 |
| 기록 | 어떤 데이터 · 프롬프트 · 모델로 만들었는지 남김 |

## ✅ 오늘 배운 것

| 하고 싶은 일 | 코드 |
|---|---|
| 분석 → Excel → 차트 → 보고서 한 번에 | `agent.run_workflow(file=파일, task="분석 목적")` |
| 다른 시나리오 | `task`만 바꾸기 |
| 다른 고객 집단 | 조건으로 뽑은 새 CSV를 `file`로 |
| 워크플로 구조 보기 | `build_agent("ex10").get_graph().draw_mermaid()` |

## 📝 복습 미션

> 아래 셀에서 **`# ←` 표시가 있는 줄만** 바꿔 실행하세요. 막히면 맨 아래 **정답 셀**을 실행해 결과를 확인합니다.

**미션.** (10-5를 실행한 경우) 10-5에서 만든 주택담보대출 고객 보고서(`outputs/ex10_정기예금_가입률_향상_보고서_bank_customers_housing.docx`)를 열어 전체 고객 수와 가입률을 확인해 보세요.
**이렇게 나오면 성공!** `전체 고객 25,130명 중 1,935명이 정기예금에 가입하여 전체 가입률은 7.7%임.`

**2일 차를 마치며 이것만은 꼭!**
- [ ] 계산은 Python, 해석은 AI, 최종 판단은 사람
- [ ] 같은 에이전트 + 다른 파일 · 다른 목적 = 반복 업무 자동화
- [ ] 금융 데이터는 승인된 데이터 · 승인된 AI 환경에서만

In [ ]:
# 미션
from docx import Document

파일 = "outputs/ex10_정기예금_가입률_향상_보고서.docx"   # ← .docx 앞에 _bank_customers_housing 붙이기
for p in Document(파일).paragraphs:
    if "전체 고객" in p.text:
        print("·", p.text)

### 정답 (먼저 위 셀을 바꿔 실행해 본 뒤 확인하세요)

In [ ]:
from docx import Document

for p in Document("outputs/ex10_정기예금_가입률_향상_보고서_bank_customers_housing.docx").paragraphs:
    if "전체 고객" in p.text:
        print("·", p.text)

---
### 2일 과정 정리

| | 1일 차 | 2일 차 |
|---|---|---|
| 도구 | Colab · Pandas · Plotly · LSTM | Jupyter · Ollama · Qwen · FinanceAgent |
| 한 일 | 경영지표 분석 · 고객 가입 패턴 · 대시보드 · 금리 추세 · 예측 | 프롬프트 · Excel 자동화 · 차트 · 보고서 · 통합 자동화 |
| 핵심 | 데이터를 읽고 비교하고 의심하기 | AI에게 일을 맡기되, 숫자와 판단은 사람이 지키기 |